In [9]:
import os

In [1]:
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.docx import partition_docx
from unstructured.documents.elements import Table, Image
from pathlib import Path

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Chuyển về folder gốc project
os.chdir(Path.cwd().parent)
print(os.getcwd())  # Phải in ra folder chứa Data/, Script/, .env

c:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2


In [2]:
import json
import logging
import traceback
from datetime import datetime
from typing import Dict, List, Tuple

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [ ]:
def create_output_path(input_file_path: Path, base_output_dir: Path = Path("Partitioned_Data")) -> Path:
    """
    Create output path mirroring input structure.
    
    Example:
        Data/QCDT/QCDT_2025.pdf → Partitioned_Data/QCDT/QCDT_2025/
    
    Args:
        input_file_path: Full path to input file
        base_output_dir: Base output directory (default: Partitioned_Data)
    
    Returns:
        Path to output folder for this file
    """
    # Get relative path from Data/ folder
    data_path = Path("Data").resolve()
    relative_path = input_file_path.relative_to(data_path)
    
    # Remove file name and extension, use as folder name
    file_name_without_ext = relative_path.stem
    parent_dirs = relative_path.parent
    
    # Build output path: Partitioned_Data/{parent_dirs}/{filename}/
    output_path = base_output_dir / parent_dirs / file_name_without_ext
    
    return output_path


def save_partitioned_data(output_path: Path, elements: List) -> Tuple[int, int, int]:
    """
    Separate partitioned elements into texts, tables, images and save as JSON.
    
    Args:
        output_path: Output directory path
        elements: List of elements from partition_pdf or partition_docx
    
    Returns:
        Tuple: (count_texts, count_tables, count_images)
    """
    texts = []
    tables = []
    images = []
    
    for element in elements:
        element_type = type(element).__name__
        
        if isinstance(element, Table):
            # Extract table with metadata
            table_data = {
                "element_type": element_type,
                "text": element.text,
                "text_as_html": element.metadata.text_as_html if hasattr(element.metadata, 'text_as_html') else None,
                "image_base64": element.metadata.image_base64 if hasattr(element.metadata, 'image_base64') else None,
            }
            tables.append(table_data)
        
        elif isinstance(element, Image):
            # Extract image with base64 and metadata
            image_data = {
                "element_type": element_type,
                "image_base64": element.metadata.image_base64 if hasattr(element.metadata, 'image_base64') else None,
                "page_number": element.metadata.page_number if hasattr(element.metadata, 'page_number') else None,
            }
            images.append(image_data)
        
        else:
            # Text and other elements
            text_data = {
                "element_type": element_type,
                "text": element.text,
            }
            texts.append(text_data)
    
    # Create output directory
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Save as JSON with ensure_ascii=False for Vietnamese content
    if texts:
        with open(output_path / "texts.json", "w", encoding="utf-8") as f:
            json.dump(texts, f, ensure_ascii=False, indent=2)
    
    if tables:
        with open(output_path / "tables.json", "w", encoding="utf-8") as f:
            json.dump(tables, f, ensure_ascii=False, indent=2)
    
    if images:
        with open(output_path / "images.json", "w", encoding="utf-8") as f:
            json.dump(images, f, ensure_ascii=False, indent=2)

    with open(output_path / "partition_done.json", "w") as f:
        json.dump({"timestamp": datetime.now().isoformat()}, f)
    
    return len(texts), len(tables), len(images)


def partition_single_file(file_path: Path, base_output_dir: Path = Path("Partitioned_Data"), skip_existing: bool = True) -> bool:
    """
    Partition a single PDF or DOCX file.
    
    Args:
        file_path: Full path to the input file (.pdf or .docx)
        base_output_dir: Base output directory
        skip_existing: If True, skip if output folder contains texts.json
    
    Returns:
        bool: True if successful, False otherwise
    """
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()
    
    # Check supported formats
    if suffix not in [".pdf", ".docx"]:
        logger.warning(f"Skipping unsupported format: {file_path} (suffix: {suffix})")
        return False
    
    # Determine output path
    output_path = create_output_path(file_path, base_output_dir)
    
    # Check if already partitioned
    if skip_existing and (output_path / "partition_done.json").exists():
        logger.info(f"Already partitioned: {file_path.name}")
        return True
    
    try:
        logger.info(f"Starting partition: {file_path.name}")
        
        # Partition based on file type
        if suffix == ".pdf":
            elements = partition_pdf(
                filename=str(file_path),
                strategy="hi_res",
                infer_table_structure=True,
                extract_image_block_types=["Image", "Table"],
                extract_image_block_to_payload=True,
                languages=["vie"],
            )
        elif suffix == ".docx":
            elements = partition_docx(filename=str(file_path))
        
        # Save partitioned data
        count_texts, count_tables, count_images = save_partitioned_data(output_path, elements)
        
        logger.info(
            f"✓ Partitioned {file_path.name}: "
            f"{count_texts} texts, {count_tables} tables, {count_images} images"
        )
        return True
    
    except Exception as e:
        logger.error(f"✗ Failed to partition {file_path.name}")
        logger.error(f"Error: {str(e)}")
        logger.error(f"Traceback:\n{traceback.format_exc()}")
        return False

In [4]:
def batch_partition(
    input_dir: Path = Path("Data"),
    base_output_dir: Path = Path("Partitioned_Data"),
    skip_existing: bool = True
) -> Dict[str, int]:
    """
    Batch partition all PDF and DOCX files in input directory recursively.
    
    Args:
        input_dir: Root input directory (default: Data)
        base_output_dir: Base output directory (default: Partitioned_Data)
        skip_existing: If True, skip already partitioned files
    
    Returns:
        Dict with counts: {total_processed, total_skipped, total_failed}
    """
    input_dir = Path(input_dir)
    
    if not input_dir.exists():
        logger.error(f"Input directory does not exist: {input_dir}")
        return {"total_processed": 0, "total_skipped": 0, "total_failed": 0}
    
    # Find all PDF and DOCX files
    pdf_files = list(input_dir.rglob("*.pdf"))
    docx_files = list(input_dir.rglob("*.docx"))
    all_files = sorted(pdf_files + docx_files)
    
    if not all_files:
        logger.warning(f"No PDF or DOCX files found in {input_dir}")
        return {"total_processed": 0, "total_skipped": 0, "total_failed": 0}
    
    logger.info(f"Found {len(all_files)} files to process")
    logger.info("=" * 80)
    
    stats = {"total_processed": 0, "total_skipped": 0, "total_failed": 0}
    
    # Process files sequentially
    for idx, file_path in enumerate(all_files, 1):
        logger.info(f"[{idx}/{len(all_files)}] Processing: {file_path.relative_to(input_dir)}")
        
        output_path = create_output_path(file_path, base_output_dir)
        
        # Check skip condition before processing
        if skip_existing and (output_path / "texts.json").exists():
            stats["total_skipped"] += 1
            logger.info(f"⊘ Skipped (already partitioned)")
        else:
            # Partition file
            success = partition_single_file(file_path, base_output_dir, skip_existing=False)
            if success:
                stats["total_processed"] += 1
            else:
                stats["total_failed"] += 1
        
        logger.info("-" * 80)
    
    # Print summary
    logger.info("=" * 80)
    logger.info("BATCH PROCESSING SUMMARY")
    logger.info("=" * 80)
    logger.info(f"Total files processed: {stats['total_processed']}")
    logger.info(f"Total files skipped:   {stats['total_skipped']}")
    logger.info(f"Total files failed:    {stats['total_failed']}")
    logger.info(f"Total files found:     {len(all_files)}")
    logger.info("=" * 80)
    
    return stats

## Example 1: Partition a Single File

Modify `test_file_path` to point to any PDF or DOCX file in the `Data/` folder.

In [ ]:
# Example: Partition a single file
# Modify this path to any PDF or DOCX file in Data/ folder
test_file_path = Path(r"C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Data\QCDT\QCDT_2025.pdf")

# Uncomment to test
if test_file_path.exists():
    success = partition_single_file(test_file_path)
    print(f"Partition result: {'Success' if success else 'Failed'}")
else:
    print(f"File not found: {test_file_path}")
    print(f"Available files in Data/:")
    for f in Path("Data").rglob("*.pdf"):
        print(f"  - {f}")
    for f in Path("Data").rglob("*.docx"):
        print(f"  - {f}")

2026-05-09 23:36:21 - INFO - Starting partition: QCDT_2025.pdf
2026-05-09 23:36:22 - INFO - pikepdf C++ to Python logger bridge initialized
2026-05-09 23:36:22 - INFO - Downloading spaCy model en_core_web_sm 3.8.0 …
2026-05-09 23:36:29 - INFO - PDF text extraction failed, skip text extraction...
2026-05-09 23:36:29 - WARNING - No languages specified, defaulting to English.
2026-05-09 23:36:30 - INFO - HTTP Request: HEAD https://huggingface.co/unstructuredio/yolo_x_layout/resolve/main/yolox_l0.05.onnx "HTTP/1.1 302 Found"
2026-05-09 23:36:30 - INFO - HTTP Request: GET https://huggingface.co/api/models/unstructuredio/yolo_x_layout/xet-read-token/7680d6f857780bcf8d49916aa2e8881bd49dee3e "HTTP/1.1 200 OK"


## Example 2: Batch Partition All Files

This will recursively find all PDF and DOCX files in `Data/` and partition them sequentially.

In [ ]:
# Run batch partitioning
# Set skip_existing=True to skip files that already have texts.json
# Set skip_existing=False to reprocess all files
stats = batch_partition(
    input_dir=Path("Data"),
    base_output_dir=Path("Partitioned_Data"),
    skip_existing=True
)

## Inspect Output Structure

View the output folder structure and sample data from partitioned files.

In [ ]:
from pathlib import Path
import json

# List all partitioned output folders
output_dir = Path("Partitioned_Data")
if output_dir.exists():
    print("Partitioned output folders:")
    print("=" * 80)
    
    for folder in sorted(output_dir.rglob("*")):
        if folder.is_dir():
            # Check what files are in this folder
            files_in_folder = list(folder.glob("*.json"))
            if files_in_folder:
                rel_path = folder.relative_to(output_dir)
                print(f"\n📁 {rel_path}")
                
                for json_file in sorted(files_in_folder):
                    file_size = json_file.stat().st_size / 1024  # KB
                    with open(json_file, 'r', encoding='utf-8') as f:
                        content = json.load(f)
                        count = len(content) if isinstance(content, list) else 1
                    print(f"   - {json_file.name}: {count} items ({file_size:.1f} KB)")
    
    print("\n" + "=" * 80)
    print(f"✓ Output directory: {output_dir.resolve()}")
else:
    print("No output directory found. Run batch partitioning first.")

## Load and View Sample Data

Example of loading partitioned data from JSON files.

In [ ]:
# Example: Load data from a partitioned file
# Modify this path to any partitioned folder

example_folder = Path("Partitioned_Data") / "QCDT" / "QCDT_2025"  # Change this path

if example_folder.exists():
    print(f"📂 Loading data from: {example_folder}")
    print("=" * 80)
    
    # Load texts
    texts_file = example_folder / "texts.json"
    if texts_file.exists():
        with open(texts_file, 'r', encoding='utf-8') as f:
            texts = json.load(f)
        print(f"\n📄 Texts ({len(texts)} items):")
        print("-" * 80)
        for i, item in enumerate(texts[:2]):  # Show first 2
            print(f"[{i}] Type: {item['element_type']}")
            print(f"    Text: {item['text'][:100]}..." if len(item['text']) > 100 else f"    Text: {item['text']}")
        if len(texts) > 2:
            print(f"    ... and {len(texts) - 2} more items")
    
    # Load tables
    tables_file = example_folder / "tables.json"
    if tables_file.exists():
        with open(tables_file, 'r', encoding='utf-8') as f:
            tables = json.load(f)
        print(f"\n📊 Tables ({len(tables)} items):")
        print("-" * 80)
        for i, item in enumerate(tables[:1]):  # Show first 1
            print(f"[{i}] Type: {item['element_type']}")
            print(f"    HTML: {item['text_as_html'][:100]}..." if item['text_as_html'] and len(item['text_as_html']) > 100 else f"    HTML: {item['text_as_html']}")
        if len(tables) > 1:
            print(f"    ... and {len(tables) - 1} more items")
    
    # Load images
    images_file = example_folder / "images.json"
    if images_file.exists():
        with open(images_file, 'r', encoding='utf-8') as f:
            images = json.load(f)
        print(f"\n🖼️  Images ({len(images)} items):")
        print("-" * 80)
        for i, item in enumerate(images[:1]):  # Show first 1
            print(f"[{i}] Type: {item['element_type']}")
            print(f"    Page: {item['page_number']}")
            print(f"    Base64 length: {len(item['image_base64']) if item['image_base64'] else 0} chars")
        if len(images) > 1:
            print(f"    ... and {len(images) - 1} more items")
    
    print("\n" + "=" * 80)
else:
    print(f"Folder not found: {example_folder}")
    print("\nAvailable partitioned folders:")
    for folder in sorted(Path("Partitioned_Data").rglob("*")):
        if folder.is_dir() and list(folder.glob("*.json")):
            print(f"  - {folder.relative_to('Partitioned_Data')}")